<a href="https://colab.research.google.com/github/emmanuelmassawe/Mnist_cnn-with-pytorch/blob/main/Mnist_cnn_with_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import matplotlib.pyplot as plt
import numpy as np

In [21]:
transform = transforms.ToTensor()

In [22]:
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

In [23]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [24]:
image , label = next(iter(train_loader))
print(image.shape)
print(label.shape)

torch.Size([64, 1, 28, 28])
torch.Size([64])


In [31]:
class cnn(nn.Module):
  def __init__(self):
    super().__init__()

    self.convlayer = nn.Sequential(
        nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),
        nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2)
    )
    self.fclayer = nn.Sequential(
        nn.Linear(in_features=64*7*7, out_features=128),
        nn.ReLU(),
        nn.Linear(in_features=128, out_features=10))

  def forward(self, x):
      x = self.convlayer(x)
      x = x.view(x.size(0), -1)
      x = self.fclayer(x)
      return x


In [32]:
model =cnn()
print(model)

cnn(
  (convlayer): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fclayer): Sequential(
    (0): Linear(in_features=3136, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [33]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [34]:
model = model.to(device)

In [35]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train

In [36]:
epochs = 10
for epoch in range(epochs):
  model.train()
  running_loss = 0
  for images, labels in train_loader:
    images = images.to(device)
    labels = labels.to(device)
    output = model(images)
    loss = loss_fn(output, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    running_loss += loss.item()

  print(f'Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader)}')

Epoch 1/10, Loss: 0.18390892697488273
Epoch 2/10, Loss: 0.04943352255576562
Epoch 3/10, Loss: 0.03448171228020718
Epoch 4/10, Loss: 0.026431250298318307
Epoch 5/10, Loss: 0.020996767885903312
Epoch 6/10, Loss: 0.01479064938589477
Epoch 7/10, Loss: 0.013506240080710134
Epoch 8/10, Loss: 0.00999430304190702
Epoch 9/10, Loss: 0.0072262724624825555
Epoch 10/10, Loss: 0.007878161009261786


In [37]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
  for image,labels in test_loader:
    image = image.to(device)
    labels = labels.to(device)
    output = model(image)

    prediction = torch.argmax(output, dim=1)
    total += labels.size(0)
    correct += (prediction == labels).sum().item()

print(f'Accuracy: {100*correct/total}%')

Accuracy: 99.02%
